# MedTrack DV — Hospital Data Cleaning
Cleans `hospital_raw_data.csv` (messy export) into the analysis-ready
`hospital_cleaned.csv` used by the Power BI dashboard.

**Steps:** drop junk rows/columns → standardize text → fix mixed date formats →
fix mixed boolean encodings → clean numeric fields → strip ID prefixes →
remove duplicates → recompute date parts → save.

## 1. Load raw data
Upload `hospital_raw_data.csv` to the Colab file browser first (left sidebar → Files → upload).

In [ ]:
import pandas as pd
import numpy as np
import re

raw = pd.read_csv('hospital_raw_data.csv')
print('Raw shape:', raw.shape)
raw.head()

## 2. Drop export artifacts
Stray index column from a naive `to_csv(index=True)` export, and fully-blank rows.

In [ ]:
df = raw.copy()
df = df.drop(columns=['Unnamed: 0'])
df = df.dropna(how='all')
print('After dropping junk:', df.shape)

## 3. Standardize text columns
Strip whitespace and collapse repeated spaces before mapping to canonical values.

In [ ]:
text_cols = ['Hospital','Department','Region','Patient_Type','Gender']
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].str.replace(r'\s+', ' ', regex=True)

## 4. Map inconsistent labels to canonical values
Handles case differences, abbreviations (e.g. `Ortho`, `ER`, `I.C.U`), and typos.

In [ ]:
hospital_map = {
    'city care hospital': 'City Care Hospital',
    'green valley hospital': 'Green Valley Hospital',
    'sunrise medical center': 'Sunrise Medical Center',
    'metro health institute': 'Metro Health Institute',
    'healthplus hospital': 'HealthPlus Hospital',
}
dept_map = {
    'general medicine': 'General Medicine', 'gen medicine': 'General Medicine',
    'surgery': 'Surgery',
    'pediatrics': 'Pediatrics', 'peds': 'Pediatrics',
    'orthopedics': 'Orthopedics', 'ortho': 'Orthopedics',
    'cardiology': 'Cardiology',
    'emergency': 'Emergency', 'er': 'Emergency',
    'icu': 'ICU', 'i.c.u': 'ICU', 'i.c.u.': 'ICU',
}
patient_type_map = {
    'inpatient': 'Inpatient', 'in-patient': 'Inpatient',
    'outpatient': 'Outpatient', 'out-patient': 'Outpatient',
    'emergency': 'Emergency',
    'day care': 'Day Care', 'daycare': 'Day Care',
}
gender_map = {
    'male': 'Male', 'm': 'Male',
    'female': 'Female', 'f': 'Female',
}

def canon(series, mapping):
    return series.str.lower().map(mapping).fillna(series)

df['Hospital'] = canon(df['Hospital'], hospital_map)
df['Department'] = canon(df['Department'], dept_map)
df['Patient_Type'] = canon(df['Patient_Type'], patient_type_map)
df['Gender'] = canon(df['Gender'], gender_map)
df['Region'] = df['Region'].str.lower().str.title()

df[text_cols].nunique()

## 5. Standardize the readmission flag
Raw data mixes `True/False`, `Yes/No`, and `1/0` — normalize to a real boolean.

In [ ]:
def to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in ('true', 'yes', '1')

df['Readmitted_30Days'] = df['Readmitted_30Days'].apply(to_bool)
df['Readmitted_30Days'].value_counts()

## 6. Parse mixed date formats
Raw dates arrive in 3 different formats (ISO, US slash, `DD-Mon-YYYY`). Pandas' `format='mixed'` detects each row's format automatically.

In [ ]:
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'], format='mixed')
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'], format='mixed')
df[['Admission_Date','Discharge_Date']].head()

## 7. Clean numeric fields
Strip stray text (`'5 days'`, `'420 beds'`, `'45 yrs'`) and cast to the correct numeric type.

In [ ]:
def extract_number(v):
    if pd.isna(v):
        return np.nan
    m = re.search(r'-?\d+\.?\d*', str(v))
    return float(m.group()) if m else np.nan

df['Length_of_Stay_Days'] = df['Length_of_Stay_Days'].apply(extract_number).astype(int)
df['Total_Beds'] = df['Total_Beds'].apply(extract_number).astype(int)
df['Patient_Age'] = df['Patient_Age'].apply(extract_number).astype(int)

## 8. Clean ID columns
Strip the `ADM-` / `PT-` prefixes and cast back to integer IDs.

In [ ]:
df['Admission_ID'] = df['Admission_ID'].astype(str).str.replace('ADM-', '', regex=False).astype(int)
df['Patient_ID'] = df['Patient_ID'].astype(str).str.replace('PT-', '', regex=False).astype(int)

## 9. Remove duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows')

## 10. Recompute derived date fields
`Year`, `Month_Num`, `Month_Name` are engineered from the now-clean `Admission_Date` — this is the KPI-engineering step from the project brief.

In [ ]:
df['Year'] = df['Admission_Date'].dt.year
df['Month_Num'] = df['Admission_Date'].dt.month
df['Month_Name'] = df['Admission_Date'].dt.strftime('%b')

## 11. Final column order and sort
Matches the schema the Power BI dashboard expects.

In [ ]:
col_order = ['Admission_ID','Patient_ID','Admission_Date','Discharge_Date','Hospital','Department',
             'Region','Patient_Type','Length_of_Stay_Days','Readmitted_30Days','Patient_Age','Gender',
             'Year','Month_Num','Month_Name','Total_Beds']
df = df[col_order].sort_values('Admission_ID').reset_index(drop=True)

print('Final cleaned shape:', df.shape)
missing_pct = df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100
print(f'Missing values: {missing_pct:.2f}%  (brief requires < 2%)')
df.head()

## 12. Save the cleaned dataset

In [ ]:
df.to_csv('hospital_cleaned.csv', index=False)
print('Saved hospital_cleaned.csv —', df.shape[0], 'rows,', df.shape[1], 'columns')

## 13. (Optional) Verify against the known-clean reference
If you also have the original `hospital_admissions_dataset.csv` handy, this confirms
the cleaning reproduced it exactly — useful evidence for your QA/testing deliverable.

In [ ]:
# Uncomment and run if you've also uploaded hospital_admissions_dataset.csv to Colab
# truth = pd.read_csv('hospital_admissions_dataset.csv', parse_dates=['Admission_Date','Discharge_Date'])
# truth = truth.sort_values('Admission_ID').reset_index(drop=True)
# for c in df.columns:
#     print(c, (df[c] == truth[c]).all())